In [ ]:
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from rich import print
from rich.logging import RichHandler
from rich.table import Table
from rich.console import Console
import time
import os
import logging
import warnings 

warnings.filterwarnings("ignore")

logger = logging.getLogger()
logger.setLevel(logging.INFO)

console_handler = RichHandler()
console_handler.setLevel(logging.INFO)

file_handler = logging.FileHandler("app.log")
file_handler.setLevel(logging.INFO)

formatter = logging.Formatter(
    "%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

file_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)

console = Console()

load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_API_ENDPOINT = os.getenv("AZURE_OPENAI_API_ENDPOINT")
AZURE_OEPNAI_API_DEPLOYMENT_4O = os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O")

llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_API_ENDPOINT,
    deployment_name=AZURE_OEPNAI_API_DEPLOYMENT_4O,
    openai_api_version=AZURE_OPENAI_API_VERSION,
    openai_api_key=AZURE_OPENAI_API_KEY
)




# ChatMessageHistory

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory

history = ChatMessageHistory()

history.add_user_message("Hi, my name is Nick.")
history.add_ai_message("Hi Nick, nice to meet you! How can I assist you today?")

print(history.messages)

# MessagesPlaceholder(記憶插槽)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

print(prompt)


# RunnableWithMessageHistory

In [ ]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# 2. Prompt + memory slot
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human","{input}")
])

chain = prompt | llm

# 
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    print(f"Current history for {session_id}:")
    for msg in store[session_id].messages:
        print(type(msg).__name__,":",msg.content)

    return store[session_id]

# 4
chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

response1 = chain_with_memory.invoke(
    {"input": "My name is Nick"},
    config={"configurable": {"session_id": "user-1"}}
)

response2 = chain_with_memory.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": "user-1"}}
)

print(response1.content)

print(response2.content)


# RunnableWithMessageHistory and RunnableLambda

In [ ]:
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnableLambda
from dotenv import load_dotenv
from rich import print
from rich.logging import RichHandler
from rich.table import Table
from rich.console import Console
import time
import os
import logging
import warnings

warnings.filterwarnings("ignore")

logger = logging.getLogger()
logger.setLevel(logging.INFO)

console_handler = RichHandler()
console_handler.setLevel(logging.INFO)

file_handler = logging.FileHandler("app.log")
file_handler.setLevel(logging.INFO)

formatter = logging.Formatter(
    "%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

file_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)

console = Console()

load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_API_ENDPOINT = os.getenv("AZURE_OPENAI_API_ENDPOINT")
AZURE_OPENAI_API_DEPLOYMENT_4O = os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O")

llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_API_ENDPOINT,
    deployment_name=AZURE_OPENAI_API_DEPLOYMENT_4O,
    openai_api_version=AZURE_OPENAI_API_VERSION,
    openai_api_key=AZURE_OPENAI_API_KEY
)

history = ChatMessageHistory()

history.add_user_message("Hi, my name is Nick.")
history.add_ai_message("Hi Nick, nice to meet you! How can I assist you today?")

print(history.messages)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human","{input}")
])


store = {}

def get_session_history(session_id: str):
    # if session_id not in store:
    #     store[session_id] = ChatMessageHistory()

    # print(f"Current history for {session_id}:")
    # for msg in store[session_id].messages:
    #     print(type(msg).__name__,":",msg.content)

    # return store[session_id]
    if session_id not in store:
        store[session_id] = ChatMessageHistory()

        store[session_id].add_user_message("Hi, my name is Bob.")
        store[session_id].add_ai_message("Hi Bob, nice to meet you! How can I assist you today?")

    print(f"\n[bold yellow]=== Current History: {session_id} ===[/bold yellow]")
    for msg in store[session_id].messages:
        print(type(msg).__name__,":", msg.content)

    return store[session_id]

def debug_messages(messages):
    # print("=== Before LLM ===")
    # for msg in messages.to_messages():
    #     print(type(msg).__name__,":",msg.content)
    # return messages
    print("\n[bold cyan]=== Messages Sent to LLM ===[/bold cyan]")
    for msg in messages.to_messages():
        print(type(msg).__name__,":",msg.content)
    return messages

chain = prompt | RunnableLambda(debug_messages) | llm

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

response1 = chain_with_memory.invoke(
    {"input":"What is my name?"},
    config={"configurable": {"session_id": "user-1"}}
)

print("\n[bold green]AI Response 1:[/bold green]")
print(response1.content)

response2 = chain_with_memory.invoke(
    {"input": "Do you remember what is my name?"},
    config={"configurable": {"session_id": "user-1"}}
)

print("\n[bold green]AI Response 2:[/bold green]")
print(response2.content)

print("\n[bold magenta]=== Final Stored Memory ===[/bold magenta]")
for msg in store["user-1"].messages:
    print(type(msg).__name__,":",msg.content)

In [ ]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnableLambda
from dotenv import load_dotenv
import os

load_dotenv()

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_API_ENDPOINT"),
    deployment_name=os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O"),
    openai_api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    openai_api_key=os.getenv("AZURE_OPENAI_API_KEY")
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        # ChatMessageHistory is list.
        store[session_id] = ChatMessageHistory() 
    return store[session_id]


chain = prompt | llm

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_message_key="input",
    history_messages_key="history"
)

print("\n=== Round 1 ===")
response1 = chain_with_memory.invoke(
    {"input": "My name is Nick"},
    config={"configurable": {"session_id": "user-1"}}
)

print(response1.content)

print("\n=== Round 2 ===")
response2 = chain_with_memory.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": "user-1"}}
)
print(response2.content)

print("\n=== Round 3 (different user) ===")
response3 = chain_with_memory.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": "user-2"}}
)
print(response3.content)

print("\n=== Memory user-1 ===")
for m in store["user-1"].messages:
    print(type(m).__name__,":",m.content)





In [ ]:
store = {}

store["a"] = []

x = store["a"]

x.append(100)

print("x:",x)

print("store",store["a"])

# LangChain + SQLite Memory

In [ ]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import SQLChatMessageHistory
from dotenv import load_dotenv
import os

load_dotenv()

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_API_ENDPOINT"),
    deployment_name=os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O"),
    openai_api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    openai_api_key=os.getenv("AZURE_OPENAI_API_KEY")
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human","{input}")
])

chain = prompt | llm

DB_URL = "sqlite:///chat_memory.db"

def get_session_history(session_id: str):
    return SQLChatMessageHistory(
        session_id=session_id,
        connection_string=DB_URL
    )

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_message_key="input",
    history_messages_key="history"
)

print("\n=== Round 1 ===")
r1 = chain_with_memory.invoke(
    {"input": "My name is Nick"},
    config={"configurable": {"session_id": "user-1"}}
)
print(r1.content)

print("\n=== Round 2 ===")
r2 = chain_with_memory.invoke(
    {"input": "Do you remember my name?"},
    config={"configurable": {"session_id": "user-1"}}
)
print(r2.content)

print("\n=== Round 3 (different user) ===")
r3 = chain_with_memory.invoke(
    {"input": "Do you remember my name?"},
    config={"configurable": {"session_id": "user-2"}}
)

print(r3.content)


In [ ]:
response4 = chain_with_memory.invoke(
    {"input": "Is my name Tom?"},
    config={"configurable": {"session_id": "user-1"}}
)

print(response4.content)

# langchain + agent + tool + memory

In [ ]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.tools import tool
from langchain_core.runnables import RunnableLambda
from langchain.agents import create_agent
from dotenv import load_dotenv
import os

load_dotenv()

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_API_ENDPOINT"),
    deployment_name=os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O"),
    openai_api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    openai_api_key=os.getenv("AZURE_OPENAI_API_KEY")
)

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    return f"The weather in {city} is sunny."

tools = [get_weather]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant that can provide weather information."
)

def get_session_history(session_id: str):
    return SQLChatMessageHistory(
        session_id=session_id,
        connection_string="sqlite:///chat_memory.db"
    )

agent_with_memory = RunnableWithMessageHistory(
    agent,
    get_session_history,
    input_messages_key="messages",
    history_messages_key="history"
)

result = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="Whis is the weather in Taipei")]},
    config={"configurable": {"session_id": "user-1"}}
)

print(result["messages"][-1].content)

# LangChain Agent + Tool + SQLite Memory

In [ ]:
import os
from dotenv import load_dotenv

from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.messages import HumanMessage

load_dotenv()

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_API_ENDPOINT"),
    deployment_name=os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O"),
    openai_api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    openai_api_key=os.getenv("AZURE_OPENAI_API_KEY")
)

@tool
def get_weather(city: str) -> str:
    """Get current weather of a city."""
    fake_weather = {
        "Taipei": "Rainy, 25°C",
        "Tokyo": "Sunnay, 28°C",
        "London": "Cloudy, 18°C"
    }

    return fake_weather.get(city, f"No weather data for {city}.")

@tool
def calculator(expression: str) -> str:
    """Calculate a math expression like 8*9 or 10+20."""
    try:
        result = eval(expression)
        return str(result)
    except:
        return "Invalid expression"
    
tools = [get_weather, calculator]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a helpful AI assistant.

Rules:
1. Use get_weather when user asks about weather.
2. Use calculator when user asks math.
3. Use chat history to understand context.
"""
)

def get_session_history(session_id: str):
    return SQLChatMessageHistory(
        session_id=session_id,
        connection_string="sqlite:///chat_memory3.db"
    )

agent_with_memory = RunnableWithMessageHistory(
    agent,
    get_session_history,
    input_messages_key="messages",
    history_messages_key="history"
)

result1 = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="What's the weather in Taipei?")]},
    config={"configurable": {"session_id": "user-1"}}
)

result2 = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="What is 12 * 8?")]},
    config={"configurable": {"session_id": "user-1"}}
)

result3 = agent_with_memory.invoke(
    {"messages": [HumanMessage(content="Waht about Tokyo?")]},
    config={"configurable": {"session_id": "user-1"}}
)

print("AI:", result1["messages"][-1].content)
print("AI:", result2["messages"][-1].content)
print("AI:", result3["messages"][-1].content)



# Interactive LangChain Agent Chatbot with Tool Use and SQLite Memory

In [ ]:
import os
from dotenv import load_dotenv

from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.messages import HumanMessage

load_dotenv()

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_API_ENDPOINT"),
    deployment_name=os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O"),
    openai_api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    openai_api_key=os.getenv("AZURE_OPENAI_API_KEY")
)

@tool
def get_weather(city: str) -> str:
    """Get current weather of a city."""
    fake_weather = {
        "Taipei": "Rainy, 25°C",
        "Tokyo": "Sunny, 28°C",
        "London": "Cloudy, 18°C"
    }
    return fake_weather.get(city, f"No weather data for {city}.")


@tool
def calculator(expression: str) -> str:
    """Evaluate math expression like 12*8"""
    try:
        return str(eval(expression))
    
    except:
        return "Invalid expression"
    
tools = [get_weather, calculator]


agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a helpful assistant.

Rules:
- Use get_weather when user asks about weather.
- Use calculator when user asks math.
- Use chat history to understand context.
"""
)

def get_session_history(session_id: str):
    return SQLChatMessageHistory(
        session_id=session_id,
        connection_string="sqlite:///chat_memory4.db"
    )


agent_with_memory = RunnableWithMessageHistory(
    agent,
    get_session_history,
    input_messages_key="messages",
    history_messages_key="history"
)


def chat():
    session_id = "user-1"

    print("🤖 AI Agent ready! Type 'exit' to quit.\n")

    while True:

        user_input = input("You: ")

        if user_input.lower() in ["exit", "quit"]:
            print("Bye 👋")
            break

        result = agent_with_memory.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            config={"configurable": {"session_id": session_id}}
        )

        answer = result["messages"][-1].content
        print("AI:",answer)
        print()

if __name__ == "__main__":
    chat()


# Simple AI Image Prompt Generator Using LangChain

In [ ]:
import os
from dotenv import load_dotenv

from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()


llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_API_ENDPOINT"),
    deployment_name=os.getenv("AZURE_OPENAI_API_DEPLOYMENT_4O"),
    openai_api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    openai_api_key=os.getenv("AZURE_OPENAI_API_KEY")
)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a professional AI image prompt engineer."
     "Convert user input into a detailed English prompt for DALL-E to generate an image."
     ),
     ("human","{input}")
])

prompt_chain = prompt | llm

def generate_image(user_input: str):

    # Step 1: create prompt
    prompt = prompt_chain.invoke({"input": user_input}).content

    # Step 2: call DALL·E
    result = client.images.generate(
        model=image_deployment,
        prompt=prompt,
        size="1024x1024"
    )

    image_url = result.data[0].url

    return prompt, image_url


# =========================
# 4. CLI
# =========================
def run():
    print("🎨 DALL·E Image Generator Ready (type 'exit' to quit)\n")

    while True:
        user_input = input("Describe image: ")

        if user_input.lower() in ["exit", "quit"]:
            print("Bye 👋")
            break

        prompt, image_url = generate_image(user_input)

        print("\n🧠 Generated Prompt:\n")
        print(prompt)

        print("\n🖼️ Image URL:\n")
        print(image_url)
        print("\n" + "-" * 60 + "\n")


if __name__ == "__main__":
    run()